# Car Price Prediction: Data Training and Analysis

This notebook loads `car_data.csv`, preprocesses the dataset, trains a `RandomForestRegressor`, evaluates model performance, and analyzes feature importances and prediction results.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

sns.set(style="whitegrid")
DATA_PATH = Path('car_data.csv')
assert DATA_PATH.exists(), f'Cannot find {DATA_PATH}. Place it in the notebook folder.'

In [ ]:
df = pd.read_csv(DATA_PATH)
print('Dataset shape:', df.shape)
display(df.head())

## Data Overview

In [ ]:
print('Columns:')
print(df.columns.tolist())
print('')
display(df.describe(include='all').T)

## Preprocessing
We will use `Selling_Price` as the target and encode categorical features using one-hot encoding. Numeric columns are converted to numeric values where possible.

In [ ]:
TARGET = 'Selling_Price'
assert TARGET in df.columns, f'{TARGET} column is required.'
df[TARGET] = pd.to_numeric(df[TARGET], errors='coerce')
numeric_cols = ['Year', 'Kms_Driven', 'Mileage', 'Engine', 'Max_Power', 'Seats']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print('Missing values per column:')
display(df.isna().mean().sort_values(ascending=False))

df = df.dropna(subset=[TARGET])

categorical_cols = [c for c in df.columns if df[c].dtype == 'object' and c != TARGET]
print('Categorical columns:', categorical_cols)
X = df.drop(columns=[TARGET])
X = pd.get_dummies(X, drop_first=True)
y = df[TARGET]
print('Processed shape:', X.shape, y.shape)

### Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)

## Model Training

In [ ]:
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
joblib.dump({'model': model, 'feature_columns': X.columns.tolist()}, 'rf_car_price_model.joblib')
print('Model training completed and saved to rf_car_price_model.joblib')

## Evaluation

In [ ]:
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)
print(f'MAE: {mae:.2f}')
print(f'RMSE: {rmse:.2f}')
print(f'R2 score: {r2:.4f}')

### Prediction vs Actual

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.5, s=20, c='tab:blue', edgecolors='w')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual Selling Price')
plt.ylabel('Predicted Selling Price')
plt.title('Actual vs Predicted Selling Price')
plt.tight_layout()
plt.show()

### Residual Distribution

In [ ]:
residuals = y_test - y_pred
plt.figure(figsize=(8, 5))
sns.histplot(residuals, bins=50, kde=True, color='tab:green')
plt.xlabel('Residual (Actual - Predicted)')
plt.title('Residual Distribution')
plt.tight_layout()
plt.show()

## Feature Importance

In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns)
top_features = importances.sort_values(ascending=False).head(20)
plt.figure(figsize=(10, 8))
sns.barplot(x=top_features.values, y=top_features.index, palette='viridis')
plt.title('Top 20 Feature Importances')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()